# Lab - Vision: Read the Order Number from a Photo

Three tasks on sending Claude a picture. Customers send photos of labels, parcels and damaged items; ShopAssist reads them with the same Messages API - a picture is one more content block.

| Task | What you learn |
| --- | --- |
| 1 | Encode the file with base64 and name its media type from its bytes |
| 2 | One message, the picture block before the question; check what was read |
| 3 | A picture costs tokens: width x height / 750 - estimate, then resize |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell runs offline. The simulator cannot see: it checks image blocks the way the API does and knows this one label, so it can answer about it. The code you write is exactly the code you would write against the real API.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import base64
import re
import struct

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
QUESTION = ("This is a photo of a shipping label. What is the order number, "
            "and which city was it shipped to?")

# The customer's orders, as the store knows them.
ORDERS = {"ORD-12345678": {"item": "jacket", "customer": "Alex Morgan"}}

# The first bytes of each image format Claude accepts.
SIGNATURES = {
    b"\x89PNG\r\n\x1a\n": "image/png",
    b"\xff\xd8\xff": "image/jpeg",
    b"GIF87a": "image/gif",
    b"GIF89a": "image/gif",
}

---

## Task 1 - Encode the picture and name its type

In [ ]:
# ============================================================
# TASK 1 - Encode the picture and name its type
# ============================================================
#
# WHAT TO DO
#   Read label.png in binary mode and encode it with base64.
#   Finish media_type_of(): name the format from the file's
#   first bytes, and refuse anything that is not an image.
#
# WHY IT MATTERS
#   A picture is sent inside the JSON request, so it has to
#   become text: base64. The image block also declares a media
#   type, and the API checks it against the data - a JPEG
#   declared as PNG is a 400. Customers upload files with any
#   name, so read the type from the bytes, not from the
#   extension.
#
# WHERE TO SEE IT IN THE LECTURE
#   "We read the file in binary mode and encode it with base
#   sixty-four".
#
# HOW TO DO IT
#   Three blanks:
#
#       data.startswith(magic)   the file begins with that signature
#       "rb"                     read bytes, not text
#       raw                      the bytes to encode
# ============================================================

try:
    # BEGIN SOLUTION
    def media_type_of(data):
        for magic, media_type in SIGNATURES.items():
            if data.startswith(magic):
                return media_type
        if data[:4] == b"RIFF" and data[8:12] == b"WEBP":
            return "image/webp"
        raise ValueError("not an image format Claude accepts")

    with open("label.png", "rb") as f:
        raw = f.read()
    image_data = base64.standard_b64encode(raw).decode()
    media_type = media_type_of(raw)
    # SCAFFOLD: def media_type_of(data):
    # SCAFFOLD:     for magic, media_type in SIGNATURES.items():
    # SCAFFOLD:         if ...:   # data.startswith(magic)
    # SCAFFOLD:             return media_type
    # SCAFFOLD:     if data[:4] == b"RIFF" and data[8:12] == b"WEBP":
    # SCAFFOLD:         return "image/webp"
    # SCAFFOLD:     raise ValueError("not an image format Claude accepts")
    # SCAFFOLD:
    # SCAFFOLD: with open("label.png", ...) as f:   # "rb" - bytes, not text
    # SCAFFOLD:     raw = f.read()
    # SCAFFOLD: image_data = base64.standard_b64encode(...).decode()   # raw
    # SCAFFOLD: media_type = media_type_of(raw)
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

print(media_type, "-", len(raw), "bytes ->", len(image_data), "base64 characters")

check("encode_image", image_data=image_data, media_type=media_type,
      media_type_of=media_type_of)

---

## Task 2 - The picture goes before the question

In [ ]:
# ============================================================
# TASK 2 - The picture goes before the question
# ============================================================
#
# WHAT TO DO
#   Build one user message with two content blocks: the image
#   block, then the text block with the question. Send it, print
#   the reply, and pull the order number out of the reply.
#
# WHY IT MATTERS
#   So far a message's content was a string; it can also be a
#   list of blocks, each with a type. The picture is not an
#   attachment next to the message - it is a block inside it,
#   and Claude reads the blocks in the order you send them:
#   picture first, then the question. What Claude reads off a
#   photo is input like any other, so the application checks
#   the order number before it uses it.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The image block has a source with three fields".
#
# HOW TO DO IT
#   Three blanks:
#
#       "base64"     the source type: the data is inline
#       media_type   from Task 1
#       QUESTION     the text block, after the picture
# ============================================================

try:
    # BEGIN SOLUTION
    message = {
        "role": "user",
        "content": [
            {"type": "image", "source": {"type": "base64",
                                         "media_type": media_type,
                                         "data": image_data}},
            {"type": "text", "text": QUESTION},
        ],
    }
    # SCAFFOLD: message = {
    # SCAFFOLD:     "role": "user",
    # SCAFFOLD:     "content": [
    # SCAFFOLD:         {"type": "image", "source": {"type": ...,          # "base64"
    # SCAFFOLD:                                      "media_type": ...,    # media_type
    # SCAFFOLD:                                      "data": image_data}},
    # SCAFFOLD:         {"type": "text", "text": ...},   # QUESTION
    # SCAFFOLD:     ],
    # SCAFFOLD: }
    # END SOLUTION: replace each ... below with the value named beside it
    response = client.messages.create(
        model=MODEL, max_tokens=300, output_config={"effort": "low"},
        messages=[message])
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for block in response.content:
    if block.type == "text":
        print(block.text)

found = re.search(r"ORD-\d{8}", response.content[0].text)   # provided: check what was read
order_id = found.group(0) if found else None
print("order id read:", order_id, "| known order:", order_id in ORDERS)

check("image_block", message=message, response=response, order_id=order_id)

---

## Task 3 - A picture costs tokens

In [ ]:
# ============================================================
# TASK 3 - A picture costs tokens
# ============================================================
#
# WHAT TO DO
#   Read the label's width and height from its PNG header,
#   estimate its tokens with the lecture's formula, and compare
#   with the input tokens the request used. Then estimate a
#   full-size phone photo.
#
# WHY IT MATTERS
#   A picture is paid for like text: roughly width times height
#   divided by 750 tokens. A small label costs a few hundred; a
#   photo straight from a phone costs many thousands. Estimate
#   before sending, and resize what is bigger than the task
#   needs.
#
# WHERE TO SEE IT IN THE LECTURE
#   "A picture costs tokens like text does".
#
# HOW TO DO IT
#   Three blanks:
#
#       width * height   the pixels in the picture
#       750              pixels per token, roughly
#       4032 * 3024      a 12-megapixel phone photo
# ============================================================

width, height = struct.unpack(">II", raw[16:24])   # provided: a PNG stores its size here

try:
    # BEGIN SOLUTION
    estimate = round(width * height / 750)
    phone_estimate = round(4032 * 3024 / 750)
    # SCAFFOLD: estimate = round(... / ...)   # width * height, 750
    # SCAFFOLD: phone_estimate = round(... / 750)   # 4032 * 3024
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(f"label {width}x{height}: about {estimate} tokens; "
      f"the request used {response.usage.input_tokens} input tokens")
print(f"a phone photo: about {phone_estimate} tokens - resize it first")

check("image_cost", estimate=estimate, response=response, phone_estimate=phone_estimate)

---

## Done

A picture is a content block in the same message as the question: encode it, give it a media type that matches its bytes, put it before the text, and read the reply as always. It costs tokens like text - estimate before you send.